
# Работа с данными бизнеса в PySpark

**Дата:** 16.07.2026

**Автор:** Исмаилов Эльчин

В этом проекте вам понадобится поработать с данными сервиса Яндекс Книги, который предоставляет доступ к контенту разных форматов, включая текст, аудио и не только. Руководство сервиса хочет лучше понимать поведение пользователей: какие типы контента они выбирают, как долго его слушают или читают, а также в какие дни недели и через какие платформы (мобильное приложение, веб-версия) это происходит. Эти инсайты позволят улучшить систему рекомендаций и принимать стратегические решения по развитию продукта. Для этого вам понадобится обработать и проанализировать реальные пользовательские данные с помощью PySpark, построить агрегаты и сделать бизнес-выводы на их основе.

Затем вы решите задачи отдела аналитики: нужно проанализировать, как различается суммарное время потребления контента в выходные и будние дни, а также выяснить, для какого типа контента наблюдается такая разница — для взрослого или невзрослого

### Описание данных


Таблица `bookmate.audition` содержит данные об активности пользователей и включает столбцы:

- `audition_id` — уникальный идентификатор сессии чтения или прослушивания;

- `puid` — идентификатор пользователя;

- `usage_platform_ru` — название платформы, с помощью которой пользователь взаимодействует с контентом;

- `msk_business_dt_str` — дата и время события (строка, часовой пояс — МСК);

- `app_version` — версия приложения;

- `adult_content_flg` — значение, которое показывает, был ли контент для взрослых ( True или False );

- `hours` — длительность сессии чтения или прослушивания в часах;

- `hours_sessions_long` — длительность длинных сессий в часах;

- `kids_content_flg` — значение, которое показывает, был ли это детский контент ( True или False );

- `main_content_id` — идентификатор основного контента;

- `usage_geo_id` — идентификатор географического местоположения пользователя.



Таблица `bookmate.content` включает столбцы:

- `main_content_id` — идентификатор основного контента;

- `main_author_id` — идентификатор основного автора контента;

- `main_content_type` — тип контента: аудио, текст или другой;

- `main_content_name` — название контента;

- `main_content_duration_hours` — длительность контента в часах;

- `published_topic_title_list` — список жанров или тем контента.

## Шаг 1. Загрузка данных и знакомство с ними

In [1]:
# Импортируем необходимые библиотеки и функции PySpark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Инициализация сессии
spark = SparkSession.builder.appName("YandexBooksAnalysis").getOrCreate()

audition = spark.read.csv("audition.csv", header=False, inferSchema=True)
content = spark.read.csv("content.csv", header=False, inferSchema=True)

# Исправляем audition_df
audition_df = audition.withColumn(
    "usage_geo_id", F.concat_ws(", ", F.col("_c11"), F.col("_c12"))
).select(
    F.col("_c0").alias("puid"),
    F.col("_c2").alias("audition_id"),
    F.col("_c3").alias("usage_platform_ru"),
    F.col("_c4").alias("msk_business_dt_str"),
    F.col("_c5").alias("app_version"),
    F.col("_c6").alias("adult_content_flg"),
    F.col("_c7").alias("hours"),
    F.col("_c8").alias("hours_sessions_long"),
    F.col("_c9").alias("kids_content_flg"),
    F.col("_c10").alias("main_content_id"),
    "usage_geo_id"
)

# Исправляем content_df
content_df = content.select(
    F.col("_c0").alias("main_content_id"),
    F.col("_c1").alias("main_content_type"),
    F.col("_c2").alias("main_content_name"),
    F.col("_c3").alias("main_content_duration_hours"),
    F.col("_c4").alias("published_topic_title_list"),
    F.col("_c5").alias("main_author_id")
)

print("Таблица audition_df (первые 10 строк)")
audition_df.show(10, truncate=False)

print("Таблица content_df (первые 10 строк)")
content_df.show(10, truncate=False)

# Проверяем типы столбцов и выводим схему
print("Схема audition_df")
audition_df.printSchema()

print("Схема content_df")
content_df.printSchema()

# Проверяем объём данных
audition_count = audition_df.count()
content_count = content_df.count()

print(f"Количество строк в audition_df: {audition_count}")
print(f"Количество строк в content_df: {content_count}")

Таблица audition_df (первые 10 строк)


+-----+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+----------------------------+
|puid |audition_id                         |usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours             |hours_sessions_long|kids_content_flg|main_content_id|usage_geo_id                |
+-----+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+----------------------------+
|162  |68296628-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |2024-11-26         |null       |false            |0.0377777777777777|0.0377777777777777 |true            |oCURrBKV       |Алматы, Казахстан           |
|213  |682966dc-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |2024-11-26         |null       |false            |8.333333333333E-4 |0.0          

Количество строк в audition_df: 1002896
Количество строк в content_df: 31668


### Промежуточные выводы

**Анализ структуры и объема данных:**
*   Таблица `audition_df` содержит **1 002 896** записей, а `content_df` — **31 668**. Такое соотношение характерно для классической схемы "звезда": `audition_df` является таблицей фактов (событийные данные о сессиях), а `content_df` — таблицей измерений (справочник уникального контента).
*   Схема данных успешно приведена к требуемому виду. Геолокация корректно объединена в единый столбец `usage_geo_id` (например, "Алматы, Казахстан"), что упрощает дальнейший географический анализ.

## Шаг 2. Трансформация и преобразование таблиц

In [2]:
# 1. Выбираем нужные столбцы и создаем minutes_sessions_long
df_step2_1 = audition_df.select(
    "puid", 
    "hours_sessions_long",
    (F.col("hours_sessions_long") * 60).cast("int").alias("minutes_sessions_long")
)

print("Первые 10 строк после расчета минут")
df_step2_1.show(10)

# 2. Добавляем столбец is_weekend
df_step2_2 = (
    audition_df.select(
        "puid",
        "hours_sessions_long",
        (F.col("hours_sessions_long") * 60).cast("int").alias("minutes_sessions_long"),
        "msk_business_dt_str",
        "adult_content_flg"
    )
    .withColumn("day_of_week", F.dayofweek(F.to_timestamp(F.col("msk_business_dt_str"))))
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)) # 1 = Воскресенье, 7 = Суббота
    .drop("day_of_week") # Удаляем вспомогательный столбец
)

df_step2_2.show(10)

# 3. Рассчитываем суммы для выходных и будних дней
df_weekend_agg = (
    df_step2_2
    .groupBy("is_weekend")
    .agg(F.sum("minutes_sessions_long").alias("total_minutes"))
    .orderBy("is_weekend")
)

print("Сумма минут по выходным/будням")
df_weekend_agg.show()

# 4. Анализ для взрослого и невзрослого контента
df_adult_agg = (
    df_step2_2
    .filter(F.col("adult_content_flg").isNotNull())
    .groupBy("adult_content_flg", "is_weekend")
    .agg(F.sum("minutes_sessions_long").alias("total_minutes"))
    .orderBy("adult_content_flg", "is_weekend")
)

print("Сумма минут по типу контента и дням недели")
df_adult_agg.show()

Первые 10 строк после расчета минут


+-----+-------------------+---------------------+
| puid|hours_sessions_long|minutes_sessions_long|
+-----+-------------------+---------------------+
|  162| 0.0377777777777777|                    2|
|  213|                0.0|                    0|
|   63|                0.0|                    0|
|    2|                0.0|                    0|
|   28|                0.0|                    0|
|   38| 0.4890794444444444|                   29|
|11162|          0.1847275|                   11|
|11119| 1.4530555555555555|                   87|
|   47| 0.5399818181818182|                   32|
|  194|                0.0|                    0|
+-----+-------------------+---------------------+
only showing top 10 rows



+-----+-------------------+---------------------+-------------------+-----------------+----------+
| puid|hours_sessions_long|minutes_sessions_long|msk_business_dt_str|adult_content_flg|is_weekend|
+-----+-------------------+---------------------+-------------------+-----------------+----------+
|  162| 0.0377777777777777|                    2|         2024-11-26|            false|     false|
|  213|                0.0|                    0|         2024-11-26|            false|     false|
|   63|                0.0|                    0|         2024-11-26|            false|     false|
|    2|                0.0|                    0|         2024-11-26|             true|     false|
|   28|                0.0|                    0|         2024-11-26|            false|     false|
|   38| 0.4890794444444444|                   29|         2024-11-26|             true|     false|
|11162|          0.1847275|                   11|         2024-11-26|             true|     false|
|11119| 1.

+----------+-------------+
|is_weekend|total_minutes|
+----------+-------------+
|     false|     17995153|
|      true|      6598993|
+----------+-------------+

Сумма минут по типу контента и дням недели


+-----------------+----------+-------------+
|adult_content_flg|is_weekend|total_minutes|
+-----------------+----------+-------------+
|            false|     false|      3369978|
|            false|      true|      1401035|
|             true|     false|     14625175|
|             true|      true|      5197958|
+-----------------+----------+-------------+



### Промежуточные выводы

**Анализ потребления контента по времени и типу:**
1. **Будни vs Выходные:** Суммарное время потребления контента в будние дни (`is_weekend = false`) составляет **17 995 153 минуты**, что значительно превышает показатель выходных дней (**6 598 993 минуты**). Это нетривиальный инсайт: аудитория сервиса активно потребляет контент в рабочие дни (возможно, во время commuting/поездок на работу или в фоновом режиме).
2. **Взрослый vs Невзрослый контент:** 
   * Контент с пометкой "для взрослых" (`adult_content_flg = true`) является абсолютным драйвером вовлеченности: **14 625 175 минут** в будни и **5 197 958 минут** в выходные.
   * Невзрослый контент (`adult_content_flg = false`) потребляется существенно меньше: **3 369 978 минут** в будни и **1 401 035 минут** в выходные.
   * *Гипотеза:* Под "взрослым" контентом в данной выборке преобладает серьезная художественная литература, нон-фикшн или длинные аудиокниги, которые пользователи слушают длительными сессиями в течение рабочей недели.

## Шаг 3. Соединение таблиц

In [3]:
# 1. Объединяем таблицы по main_content_id
joined_df = audition_df.join(content_df, on="main_content_id", how="inner")

joined_count = joined_df.count()
print(f"Количество строк после объединения: {joined_count}")

# 2. Удаляем лишние столбцы
joined_df = joined_df.drop("main_author_id", "app_version", "usage_geo_id")

print("Первые 10 строк объединенной таблицы")
joined_df.show(10, truncate=False)

# 3. Считаем уникальных пользователей и сравниваем
orig_puid_count = audition_df.select("puid").distinct().count()
joined_puid_count = joined_df.select("puid").distinct().count()

print(f"Уникальных пользователей в исходной audition_df: {orig_puid_count}")
print(f"Уникальных пользователей в joined_df: {joined_puid_count}")

# 4. Выводим уникальные значения main_content_type через collect()
unique_content_types = joined_df.select("main_content_type").distinct().collect()

for type in unique_content_types:
    print(type["main_content_type"])

Количество строк после объединения: 996565
Первые 10 строк объединенной таблицы


+---------------+-----+------------------------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+-----------------+---------------------------------------------------------------------------------------------------------+---------------------------+----------------------------------------------------------------------------------------------------+
|main_content_id|puid |audition_id                         |usage_platform_ru|msk_business_dt_str|adult_content_flg|hours             |hours_sessions_long|kids_content_flg|main_content_type|main_content_name                                                                                        |main_content_duration_hours|published_topic_title_list                                                                          |
+---------------+-----+------------------------------------+-----------------+-------------------+-----------------+------------------+-------------

Уникальных пользователей в исходной audition_df: 2576
Уникальных пользователей в joined_df: 2574


Audiobook
Book
Comicbook


### Промежуточные выводы

**Качество данных и результат объединения (Join):**
*   После `INNER JOIN` количество строк сократилось с 1 002 896 до **996 565**. Потеряно **6 331 строка (≈0.6%)**. Это означает, что для этих сессий не нашлось соответствующего `main_content_id` в справочнике контента (возможно, контент был удален из каталога или произошла ошибка в ETL-процессе).
*   Количество уникальных пользователей (`puid`) сократилось с **2 576** до **2 574**. Потеря **2 пользователей** говорит о том, что у них *все* сессии были связаны с "осиротевшим" контентом, которого нет в справочнике.
*   В сервисе представлено **3 формата** контента: `Audiobook` (аудиокниги), `Book` (текстовые книги) и `Comicbook` (комиксы). Судя по выборке, аудиокниги доминируют в сессиях прослушивания.

### Итоговые выводы и рекомендации

На основе проведенного анализа данных сервиса Яндекс Книги сформулированы следующие выводы и рекомендации для продуктовой команды:

1. **Смещение фокуса на будние дни:** Вопреки классическому паттерну "досуг в выходные", основная масса потребления контента (более 73% времени) приходится на будние дни. 
   * *Рекомендация:* Разработать и протестировать фичи для "будничного" потребления: плейлисты "Для дороги на работу", умные уведомления в часы пик (утро/вечер будней), улучшение функции офлайн-скачивания для метро.
2. **Доминирование "взрослого" контента:** Контент с флагом `adult_content_flg = true` генерирует более 80% всего времени вовлеченности. 
   * *Рекомендация:* Усилить рекомендательную систему именно для этой категории в будние дни. Провести когортный анализ, чтобы понять, является ли это особенностью конкретной выборки или устойчивым трендом всей базы. Если это длинные аудиокниги, стоит продвигать подписку с акцентом на "книги, которые заменят подкасты в дороге".
3. **Развитие нишевых форматов:** Наличие `Comicbook` и `Book` наряду с доминирующим `Audiobook` показывает диверсификацию продукта. 
   * *Рекомендация:* Проанализировать конверсию из текстовой книги в аудиокнигу для одних и тех же тайтлов, чтобы стимулировать апсейл на более дорогую или длительную по потреблению аудиоподписку.